[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/05_themen_extrahieren.ipynb)

# Sitzung 5 — Themen extrahieren

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Bisher wisst ihr, *dass* Leute (un)zufrieden sind. Heute: *worüber* reden sie? **Themen** — Akku, Preis, App, Klang. Das ist, was eine Produktmanagerin wirklich braucht, um zu entscheiden, was als Nächstes verbessert wird.

## 0. Setup & Daten

In [ ]:
import random, re
from collections import Counter
from datetime import date, timedelta
print('Fertig.')

In [ ]:
import random
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"

POS = ["der Klang ist hervorragend", "satte Bässe", "die Geräuschunterdrückung ist top",
       "der Akku hält den ganzen Tag", "sitzt super bequem", "Bluetooth verbindet sofort",
       "top verarbeitet", "klasse für den Preis", "die App ist übersichtlich",
       "die Passform ist perfekt", "der Sound ist klar und ausgewogen"]
NEG = ["die App stürzt ständig ab", "der rechte Ohrhörer lädt nicht mehr",
       "die Geräuschunterdrückung rauscht", "viel zu teuer", "die Touch-Steuerung reagiert kaum",
       "das Case wirkt billig", "der Akku ist nach einer Stunde leer",
       "die Verbindung bricht ab", "sie fallen leicht aus dem Ohr", "der Bass ist matschig"]

POS_OPENERS = ["Bin begeistert:", "Wirklich gut:", "Kann ich empfehlen –", "Top Kauf.",
               "Sehr zufrieden:", "Absolute Kaufempfehlung.", "Ich liebe sie:",
               "Klare Sache:", "Rundum gelungen:", "Volle Punktzahl:", "Endlich zufrieden:",
               "Was soll ich sagen –", "Genau richtig:", "Bestellung hat sich gelohnt:"]
NEG_OPENERS = ["Enttäuschend:", "Leider schlecht:", "Finger weg –", "Bin frustriert:",
               "Nicht zu empfehlen.", "Schade um das Geld:", "Ärgerlich:",
               "Reklamiert:", "Bin raus:", "Nie wieder:", "Herbe Enttäuschung:",
               "Das war nichts:", "Zurückgeschickt:", "Vorsicht:"]
NEU_TEMPLATES = ["Ganz okay, {a}, aber nichts Besonderes.",
                 "Erfüllt seinen Zweck. {a_cap}.",
                 "Durchschnittlich. {a_cap}, mehr nicht.",
                 "Habe sie seit Kurzem, {a} – kann noch nicht viel sagen."]
SARCASTIC = ["Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
             "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
             "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
             "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
             "Klasse, nach einer Woche nur noch Rauschen. Wirklich durchdacht.",
             "Perfekt, der linke fällt ständig raus. Genau mein Wunsch.",
             "Herrlich, die Verbindung bricht alle fünf Minuten ab. Danke auch.",
             "Sensationell, das Case bricht beim ersten Öffnen. Qualität eben.",
             "Bravo, nach dem Update ist die Hälfte der Funktionen weg.",
             "Fantastisch leise – weil nach zwei Tagen einfach tot."]
FAKE = ["BESTES PRODUKT EVER!!! Kauft bei www.super-deals-guenstig.example!!!",
        "5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
        "Gutschein Code NIMBUS100 auf meiner Seite jetzt klicken!!!",
        "amazing product best quality buy now discount link in profile",
        "TOP TOP TOP unbedingt kaufen billigster preis hier klicken",
        "gratis versand nur heute!!! rabattcode DEAL22 einlösen!!!",
        "beste kopfhoerer der welt jetzt zuschlagen link im profil",
        "WOW einfach WOW kaufen kaufen kaufen bester preis garantiert",
        "unglaublich guenstig hier klicken und sparen sparen sparen",
        "mega angebot heute -70% nur ueber meinen link!!!"]
ENGLISH = [("Sound quality is great but the app is a disaster.", "mixed"),
           ("Battery life is amazing, best earbuds I have owned.", "positive"),
           ("Stopped working after a week, very disappointed.", "negative"),
           ("Comfortable fit and clear sound, happy with the purchase.", "positive"),
           ("The noise cancelling is weak and the case feels cheap.", "negative")]
JUNK = ["", "   ", ".", "???", "kein kommentar", "-", "n/a", "...", "!!", "??", "keine angabe", "test"]

def _pos(rng):
    o = rng.choice(POS_OPENERS); a = rng.sample(POS, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "positive"
def _neg(rng):
    o = rng.choice(NEG_OPENERS); a = rng.sample(NEG, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "negative"
def _mixed(rng):
    p = rng.choice(POS); n = rng.choice(NEG)
    conn = rng.choice([" - aber ", ", allerdings ", ". Leider ", ", jedoch "])
    return f"{p[0].upper()+p[1:]}{conn}{n}.", "mixed"
def _neutral(rng):
    a = rng.choice(POS + NEG)
    return rng.choice(NEU_TEMPLATES).format(a=a, a_cap=a[0].upper()+a[1:]), "neutral"
def _rating_for(truth, rng):
    return rng.choice({"positive":[4,5,5],"negative":[1,1,2],"mixed":[2,3,4],
                       "neutral":[3,3,4],"fake":[5,5]}.get(truth,[1,3,5]))

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:   text, truth = _pos(rng)
        elif r < 0.60: text, truth = _neg(rng)
        elif r < 0.72: text, truth = _mixed(rng)
        elif r < 0.80: text, truth = _neutral(rng)
        elif r < 0.88: text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.93: text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98: text, truth = rng.choice(ENGLISH)
        else:          text, truth = rng.choice(JUNK), "junk"
        day = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day)).isoformat(),
            "product": PRODUCT,
            "rating": _rating_for(truth, rng),
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": text in SARCASTIC,
            "is_fake": text in FAKE,
        })
    if n > 20:
        for j, src in enumerate([5, 12, 30]):
            rows.append(dict(rows[src], review_id=f"R{n+j:04d}"))
    rng.shuffle(rows)
    return rows

In [ ]:
reviews = generate(300)
print(f'{len(reviews)} Bewertungen.')

## 1. Von Polarität zu Substanz

„40% negativ“ ist eine Zahl. Aber sie sagt nicht, **was** das Problem ist. Erst Themen machen daraus eine Handlungsanweisung:

- *„30% der negativen Bewertungen betreffen den **Akku**“* → Akku verbessern.
- *„Der **Preis** wird oft kritisiert, aber der **Klang** gelobt“* → Positionierung.

Wir arbeiten mit einer **festen Themen-Liste** — nur so lassen sich Themen zählen und vergleichen. (Freitext-Themen könnte man nicht aggregieren.)

In [ ]:
THEMEN = ['Klang', 'Akku', 'Geräuschunterdrückung', 'App',
          'Verbindung', 'Komfort', 'Preis', 'Verarbeitung']
print('Themen-Vokabular:', THEMEN)

## 2. Der naive Weg: Stichwort-Regeln

Der erste Reflex: für jedes Thema ein paar **Stichwörter** definieren und danach suchen. Simpel, kein LLM nötig. Bauen wir das:

In [ ]:
STICHWOERTER = {
    'Klang': ['klang', 'sound', 'bass', 'bässe'],
    'Akku': ['akku', 'batterie', 'laden', 'lädt'],
    'Geräuschunterdrückung': ['geräuschunterdrückung', 'rauscht', 'noise'],
    'App': ['app'],
    'Verbindung': ['bluetooth', 'verbindung'],
    'Komfort': ['bequem', 'sitzt', 'passform', 'ohr'],
    'Preis': ['preis', 'teuer', 'euro', 'geld'],
    'Verarbeitung': ['verarbeitet', 'case', 'billig', 'qualität'],
}

def themen_regeln(text):
    t = text.lower()
    return [thema for thema, woerter in STICHWOERTER.items()
            if any(w in t for w in woerter)]

for r in reviews[:6]:
    print(themen_regeln(r['text']), '|', r['text'][:55])

## 3. Wo die Regeln brechen

Sieht erstmal okay aus. Aber Stichwörter sind **dumm** — sie verstehen keine Bedeutung. Probiert diese Fälle:

In [ ]:
beispiele = [
    'Die Batterie ist nach einer Stunde leer.',
    'Der Ton ist grandios.',
    'Die Passform könnte nicht besser sein.',
    'Preislich okay, aber der Sound enttäuscht.',
    'Kein Akku-Problem, hält ewig.',
]
for b in beispiele:
    print(themen_regeln(b), '|', b)

> ✏️ **Fällt euch auf?** „Der **Ton** ist grandios“ → *kein* Thema erkannt, weil „ton“ nicht in der Liste steht. Synonyme, Umschreibungen, Kontext — Stichwörter scheitern daran. Man müsste die Liste **endlos** pflegen.

## 4. Der LLM-Weg

Ein LLM **versteht** die Bewertung — es braucht keine Stichwort-Liste, nur die Themen-Auswahl. Hier die Funktion (Mock = die Regeln von oben; echt = LLM):

In [ ]:
def _mock_themen(text):
    # Der Mock IST der naive Regel-Ansatz von oben.
    return themen_regeln(text)

def _real_themen(text, client, model='claude-sonnet-5'):
    import json
    prompt = (f'Welche dieser Themen kommen in der Bewertung vor? '
              f'Themen: {THEMEN}. Antworte NUR als JSON-Liste, z.B. ["Akku","Preis"].\n\n'
              f'Bewertung: """{text}"""')
    resp = client.messages.create(model=model, max_tokens=150,
                                  messages=[{'role':'user','content':prompt}])
    raw = next((b.text for b in resp.content if hasattr(b,'text')), '[]')
    m = re.search(r'\[.*\]', raw, re.DOTALL)
    return json.loads(m.group(0)) if m else []

def extrahiere_themen(text, use_real=False, client=None):
    return _real_themen(text, client) if use_real else _mock_themen(text)

print('Funktion bereit. Mock = Regeln, echt = LLM.')

> 🎓 **Live (Vortragende:r):** Dieselben Trick-Beispiele von oben durch das **echte** LLM schicken (`use_real=True`). „Der Ton ist grandios“ → das LLM erkennt **Klang**, ohne dass „ton“ je in einer Liste stand. Das ist der Unterschied zwischen *Stichwort* und *Bedeutung*.

## 5. Themen über alle Bewertungen

Jetzt der eigentliche Nutzen: **welche Themen kommen am häufigsten vor?** (Über den Mock, damit es key-frei läuft.)

In [ ]:
alle_themen = Counter()
for r in reviews:
    for thema in extrahiere_themen(r['text']):
        alle_themen[thema] += 1

print('Häufigste Themen:')
for thema, c in alle_themen.most_common():
    print(f'  {thema:22} {c}')

## 6. Eure Aufgabe

Interessant wird es, wenn man Themen mit Sentiment **kreuzt**: worüber wird *negativ* geredet? Das sind die Baustellen.

In [ ]:
negativ_themen = Counter()
for r in reviews:
    if r['rating'] <= 2:
        for thema in extrahiere_themen(r['text']):
            negativ_themen[thema] += 1
print('Top-Beschwerde-Themen:')
for thema, c in negativ_themen.most_common(5):
    print(f'  {thema:22} {c}')

> ✏️ **Eure Aufgabe:** Vergleicht die Themen in **guten** (Sterne >= 4) und **schlechten** Bewertungen. Worüber wird gelobt, worüber geschimpft? Das ist eine echte Produkt-Erkenntnis.

## Der tiefere Punkt: „Bedeutung hat eine Geometrie“

Woher *weiß* ein LLM, dass „Ton“ und „Klang“ dasselbe Thema sind, ohne Stichwort-Liste? Die Idee dahinter: **Wörter und Sätze werden in Zahlen übersetzt** — sogenannte *Embeddings* — und zwar so, dass **ähnliche Bedeutungen nahe beieinander liegen**.

Stellt euch eine Landkarte der Bedeutungen vor:

- „Akku“ und „Batterie“ liegen dicht beieinander.
- „Klang“ und „Ton“ ebenfalls.
- „Preis“ liegt weit weg von „Komfort“.

So wird *Bedeutung* zu *Nähe im Raum* — und Nähe kann man rechnen. Das ist die Grundlage vieler moderner KI-Anwendungen.

> 💡 **Kernpunkt:** Nicht Stichwörter, sondern **Bedeutung als Geometrie**. Nächste Woche rechnen wir das *echt* aus — und ihr seht, wie „Akku“ und „Batterie“ im Zahlenraum zusammenrücken. *(Embeddings, Teil 2.)*

## 7. Geschafft — und Ausblick

Aus „40% negativ“ ist geworden: *„worüber* die Leute reden — und wo die Baustellen sind.“ Das Werkzeug wird zum echten Insight-Tool.

**Nächste Woche (11.11):** verlässliche, strukturierte Ausgabe (JSON) — und Embeddings zum Anfassen.